### Código fornecido para preparar o ambiente.

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuração visual
sns.set_theme(style="whitegrid", context="talk")

print("⏳ A gerar a base de dados da FitTrack Analytics (Corrigida)...")
np.random.seed(42)

n = 2500
user_id = range(1001, 1001 + n)
signup_date = pd.date_range(start='2021-01-01', periods=n, freq='12h')
device_info = np.random.choice(['Apple Watch v5', 'Garmin Forerunner', 'Smartphone iOS', 'Android Phone', 'Mi Band 6'], n)

# Lógica de negócio corrigida: O preço depende do tipo de plano
subscription = np.random.choice(['Free', 'Premium', 'Pro'], n, p=[0.5, 0.3, 0.2])

monthly_fee = []
for sub in subscription:
    if sub == 'Free':
        # O sistema às vezes registra 0.00, às vezes escreve a palavra 'Free'
        monthly_fee.append(np.random.choice(['$ 0.00', 'Free'], p=[0.8, 0.2]))
    elif sub == 'Premium':
        monthly_fee.append('$ 9.99')
    else:
        monthly_fee.append('$ 19.99')

age = np.random.normal(35, 10, n).round()
daily_steps = np.random.normal(8000, 3000, n).round()
sleep_hours = np.random.normal(7, 1.5, n).round(1)

df_fit = pd.DataFrame({
    'user_id': user_id,
    'signup_date': signup_date,
    'device_info': device_info,
    'subscription': subscription,
    'age': age,
    'daily_steps': daily_steps,
    'sleep_hours': sleep_hours,
    'monthly_fee': monthly_fee
})

# Injetando nulos propositais
df_fit.loc[np.random.choice(df_fit.index, 150, replace=False), 'age'] = np.nan
df_fit.loc[np.random.choice(df_fit.index, 300, replace=False), 'monthly_fee'] = np.nan

# Injetando a string "NaN" propositalmente para confundir
df_fit.loc[np.random.choice(df_fit.index, 50, replace=False), 'monthly_fee'] = "NaN"

# Outliers extremos
df_fit.loc[10:15, 'sleep_hours'] = [25.0, 30.5, 45.0, -5.0, 50.2, 28.0]
df_fit.loc[100:105, 'daily_steps'] = [150000, 200000, -100, 180000, 500, 165000]

df_fit.to_csv('health_app_data.csv', index=False)
print("✅ Arquivo 'health_app_data.csv' atualizado no disco local!")

⏳ A gerar a base de dados da FitTrack Analytics (Corrigida)...
✅ Arquivo 'health_app_data.csv' atualizado no disco local!


# 🏃‍♂️ Desafio de Negócios: FitTrack Analytics
### Contexto: 
Você trabalha para a FitTrack, uma startup de monitoramento de saúde. O time de engenharia extraiu um arquivo do banco de dados com os perfis dos usuários, mas avisou que o dado está "cru". O CEO precisa de um relatório exploratório para amanhã.

Carregue o CSV gerado e resolva as demandas abaixo.

## As Requisições (Demandas):


#### 1. Limpeza Financeira (Strings e Nulos)
O Problema:
 - A coluna monthly_fee (Mensalidade) está como texto, possui símbolos de dólar ($), contém a palavra "Free", além de possuir valores nulos (NaN) e a string "NaN".

A Demanda: 
 - O financeiro precisa que essa coluna seja numérica (float).
 - Trate as strings para que os números possam ser extraídos.
 - Converta a coluna para o tipo correto.
 - Para os valores que ficarem nulos (faltantes) após a conversão, faça a imputação utilizando a mediana da mensalidade, calculada agrupando pelo tipo de assinatura (subscription).


In [ ]:
df_fit = pd.read_csv(r"C:\Users\leand\Documents\Data-Science-Exercises\FitTrack_EDA\health_app_data.csv", parse_dates = True)

: 

: 

In [ ]:
df_fit.info()

<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   user_id       2500 non-null   int64  
 1   signup_date   2500 non-null   str    
 2   device_info   2500 non-null   str    
 3   subscription  2500 non-null   str    
 4   age           2350 non-null   float64
 5   daily_steps   2500 non-null   float64
 6   sleep_hours   2500 non-null   float64
 7   monthly_fee   2157 non-null   str    
dtypes: float64(3), int64(1), str(4)
memory usage: 156.4 KB


: 

: 

In [ ]:
df_fit["monthly_fee"].value_counts()

monthly_fee
$ 0.00     867
$ 9.99     644
$ 19.99    421
Free       225
Name: count, dtype: int64

: 

: 

In [ ]:
df_fit["monthly_fee"] = df_fit["monthly_fee"].replace("^\$|Free|" "", "", regex = True)

: 

: 

In [ ]:
df_fit["monthly_fee"] = pd.to_numeric(df_fit["monthly_fee"])

: 

: 

In [ ]:
df_fit["monthly_fee"] = df_fit.groupby("subscription")["monthly_fee"].transform(lambda x: x.fillna(x.median()))

: 

: 

In [ ]:
df_fit.groupby("subscription")["monthly_fee"].value_counts()

subscription  monthly_fee
Free          0.00           1262
Premium       9.99            747
Pro           19.99           491
Name: count, dtype: int64

: 

: 

In [ ]:
df_fit.info()

<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   user_id       2500 non-null   int64  
 1   signup_date   2500 non-null   str    
 2   device_info   2500 non-null   str    
 3   subscription  2500 non-null   str    
 4   age           2350 non-null   float64
 5   daily_steps   2500 non-null   float64
 6   sleep_hours   2500 non-null   float64
 7   monthly_fee   2500 non-null   float64
dtypes: float64(4), int64(1), str(3)
memory usage: 156.4 KB


: 

: 

#### 2. Higienização Clínica (Tratamento de Outliers)
O Problema: 
 - A equipe médica relatou que o aplicativo registrou horas de sono impossíveis (como 45 horas num dia ou valores negativos) e contagens de passos irreais devido a bugs nos relógios.

A Demanda: 
 - Utilize a regra estatística do Intervalo Interquartil (IQR) para identificar limites inferiores e superiores exclusivamente para a coluna sleep_hours. 
 - Remova do DataFrame todos os usuários que possuam outliers nessa coluna.

#### 3. Comportamento e Retenção (Datas e Categorias)
A Demanda: 
 - O marketing quer entender os padrões de cadastro e categorizar o nível de atividade da base (agora já limpa do passo 2).
 - Descubra e imprima qual é o dia da semana com o maior volume absoluto de cadastros (signup_date).
 - Crie uma nova coluna chamada activity_level. Ela deve categorizar os usuários baseado na coluna daily_steps em três rótulos: "Sedentary" (Até 5000 passos), "Active" (De 5000 até 10000 passos) e "Athlete" (Acima de 10000 passos).
 - Apresente uma tabela de frequência cruzada (contingência) mostrando a relação entre a categoria de atividade que você acabou de criar e o tipo de plano de assinatura (subscription).


: 

: 

#### 4. Segmentação de Dispositivos (Text Mining)
A Demanda: 
 - A equipe de produto quer saber quantos usuários utilizam dispositivos de pulso (relógios/pulseiras) em comparação com celulares.
 - Usando a coluna device_info, encontre todas as linhas que contenham as palavras "Watch", "Band" ou "Forerunner" (independente de ser maiúscula ou minúscula). Quantos usuários usam esse tipo de dispositivo vestível?

: 

: 

#### 5. O Deck da Apresentação (Visualização Avançada)
A Demanda: 
 - O CEO pediu dois gráficos profissionais para a apresentação de resultados, que evidenciem padrões sem a necessidade de olhar para os números.
 - Gráfico A: Um Mapa de Calor (Heatmap) que exiba claramente o grau de correlação linear entre todas as variáveis estritamente numéricas do dataset limpo.
 - Gráfico B: Um Gráfico de Densidade (KDE Plot) das horas de sono (sleep_hours), onde as curvas sejam separadas e coloridas pelo tipo de assinatura (subscription). Queremos ver se quem paga planos mais caros dorme melhor.